# Error Taxonomy Analysis - All Simulated Failed Episodes

Two-round workflow:
1. **Round 1 - Triage**: Load all failed episodes, display key context (GT reason, predicted reason, Loc, Exp, captions summary, predicted error type), and let you assign a taxonomy category to each with a quick dropdown.
2. **Round 2 - Deep dive**: Re-examine episodes that were marked `needs_review` in round 1 with full scene-graph and caption inspection.

### Error Mode Taxonomy (Root-Cause)

Each code identifies **where in the pipeline** information or reasoning failed - not just a symptom already captured by Loc/Exp/Co-plan.

| Code | Layer | Root cause |
|------|-------|------------|
| `PER-MISSING` | Perception | Relevant state never captured in **any** frame - absent from both L1 and L2 |
| `PER-SPATIAL` | Perception | Perception assigned the **wrong spatial relationship** (wrong preposition / wrong relative object) - object detected but wrongly located |
| `CTX-HIDDEN` | Context | State captured in L1 (non-action frames) but **absent from L2** - LLM never saw it |
| `CTX-AFFORD` | Context | Info present in L2 but too generic/underspecified for LLM to **ground to the specific required object or precondition**; also covers task specs too ambiguous to map to a concrete affordance |
| `RSN-DIAG` | Reasoning | LLM had the context but **misidentified the failure** - wrong step, wrong cause, or complete miss at diagnosis time |
| `RSN-PLAN` | Reasoning | LLM diagnosed correctly but generated a **logically flawed or incomplete correction plan** - reasoning failure in the planning sub-task |
| `EXEC-SIM` | Execution | Simulator / physics failure; reasoning and plan were sound |
| `NEEDS-REVIEW` | Admin | Insufficient evidence; requires deeper episode inspection |
| `OTHER` | Admin | Genuine outlier that does not fit the pipeline model |

In [ ]:
# Imports
import pandas as pd
import numpy as np
import json, os
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import re as _re
import matplotlib.pyplot as plt

In [ ]:
# Constants & paths
from reflect.core.paths import sim_results_root

RESULTS_PATH    = str(sim_results_root()) + "/"
SIM_DATA_DIR    = "./datasets/sim_data/"
RUN_JSON        = os.path.join(RESULTS_PATH, "reflect_results_run4.json")
EVAL_CSV        = os.path.join(RESULTS_PATH, "exp_evaluation_run4.csv")
OUTPUT_CSV      = os.path.join(RESULTS_PATH, "error_taxonomy_all_run4.csv")

# Taxonomy codes
TAXONOMY_CODES = [
    "PER-MISSING",   # Perception: state never captured in any frame (L1 + L2 both missing)
    "PER-SPATIAL",   # Perception: wrong spatial relationship assigned (wrong preposition/relative obj)
    "CTX-HIDDEN",    # Context: captured in L1 but absent from L2 (LLM never saw it)
    "CTX-AFFORD",    # Context: info in L2 but too generic to ground to specific object/precondition
    "RSN-DIAG",      # Reasoning: LLM misidentified the failure (wrong step/cause/complete miss)
    "RSN-PLAN",      # Reasoning: correct diagnosis but flawed/incomplete correction plan
    "EXEC-SIM",      # Execution: simulator/physics failure; reasoning & plan were sound
    "NEEDS-REVIEW",  # Admin: insufficient evidence, needs deeper inspection
    "OTHER",         # Admin: genuine outlier
]

# Taxonomy descriptions
TAXONOMY_DESCRIPTIONS = {
    "PER-MISSING":  "Relevant state never captured in any frame - absent from L1 and L2.",
    "PER-SPATIAL":  "Perception assigned the wrong spatial relationship (wrong preposition / wrong relative object).",
    "CTX-HIDDEN":   "State captured in L1 (non-action frames) but absent from L2 - LLM never saw it.",
    "CTX-AFFORD":   "Info present in L2 but too generic/underspecified to ground to specific object or precondition; also covers ambiguous task specs.",
    "RSN-DIAG":     "LLM had the context but misidentified the failure - wrong step, wrong cause, or complete miss at diagnosis time.",
    "RSN-PLAN":     "LLM diagnosed correctly but generated a logically flawed or incomplete correction plan.",
    "EXEC-SIM":     "Simulator / physics failure; reasoning and plan were sound.",
    "NEEDS-REVIEW": "Insufficient evidence to assign root cause - requires deeper episode inspection.",
    "OTHER":        "Genuine outlier that does not fit the pipeline model.",
}

# Path for user-defined custom codes
CUSTOM_CODES_PATH = os.path.join(RESULTS_PATH, "custom_taxonomy_codes.json")

In [ ]:
# Load & merge data
with open(RUN_JSON) as f:
    raw = json.load(f)
run_df = pd.DataFrame(raw)

# Unfold nested dicts
reasoning = pd.json_normalize(run_df['reasoning_dict']).add_prefix('r_')
replan    = pd.json_normalize(run_df['replan_dict']).add_prefix('rp_')
correct   = pd.json_normalize(run_df['correction_dict']).add_prefix('c_')
run_df = pd.concat([run_df.drop(columns=['reasoning_dict','replan_dict','correction_dict']),
                     reasoning, replan, correct], axis=1)

# Merge human scores
scored = pd.read_csv(EVAL_CSV)
run_df = run_df.merge(scored[['episode','human_score']], on='episode', how='left')

# Load per-episode fields from dataset task.json files
def _load_task_json_field(row, field, default=''):
    task_json = os.path.join(SIM_DATA_DIR, row['task'], row['episode'], 'task.json')
    try:
        with open(task_json) as f:
            return json.load(f).get(field, default)
    except (FileNotFoundError, json.JSONDecodeError):
        return default

run_df['success_condition'] = run_df.apply(lambda r: _load_task_json_field(r, 'success_condition'), axis=1)
run_df['gt_plan'] = run_df.apply(lambda r: _load_task_json_field(r, 'actions', []), axis=1)

# Helpers
def ts_to_sec(ts):
    m, s = ts.strip().split(":")
    return int(m) * 60 + int(s)

def assess_loc(pred_list, gt_list):
    if not pred_list or not gt_list:
        return False
    try:
        p_sec = ts_to_sec(pred_list[0])
        gt_secs = [ts_to_sec(t) for t in (gt_list if isinstance(gt_list, list) else [gt_list])]
        return p_sec == gt_secs[0] if len(gt_secs) == 1 else gt_secs[0] <= p_sec <= gt_secs[-1]
    except Exception:
        return False

# Compute Loc and Exp columns
run_df['Loc'] = run_df.apply(lambda r: assess_loc(r['r_pred_failure_step'], r['r_gt_failure_step']), axis=1)
run_df['Exp'] = run_df['human_score'].fillna(0).astype(bool)

# Filter to failed episodes only
failed = run_df[run_df['c_success'] == False].copy().reset_index(drop=True)
print(f"Total failed episodes to categorize: {len(failed)}")
print(f"  Loc=True & Exp=True : {((failed['Loc']) & (failed['Exp'])).sum()}")
print(f"  Loc=True & Exp=False: {((failed['Loc']) & (~failed['Exp'])).sum()}")
print(f"  Loc=False & Exp=True: {((~failed['Loc']) & (failed['Exp'])).sum()}")
print(f"  Loc=False & Exp=False: {((~failed['Loc']) & (~failed['Exp'])).sum()}")


In [ ]:
# Suggestion function for root cause taxonomy codes based on GT failure reason, L1/L2 context, and Loc/Exp signals
# Generic object terms in success conditions that signal a grounding/affordance gap
_GENERIC_SC_TERMS = {'container', 'object', 'item', 'vessel', 'receptacle'}


def _extract_blocking_object(gt_lower):
    """Extract the object causing blocking/occlusion from GT reason text."""
    m = _re.search(r'due to (?:a |the |an )?(\w+)', gt_lower)
    if m and m.group(1) not in ('the', 'a', 'an', 'it'):
        return m.group(1)
    m = _re.search(r'(\w+)\s+(?:is\s+)?(?:blocking|occluding)', gt_lower)
    if m:
        return m.group(1)
    return None


def _has_generic_success_condition(succ_cond):
    return any(term in succ_cond.lower() for term in _GENERIC_SC_TERMS)


def _rsn_from_loc_exp(loc, exp, exp_unknown, base_reason):
    """When the LLM had the info, route to RSN-DIAG or RSN-PLAN based on Loc/Exp/CoP."""
    if exp_unknown:
        return 'NEEDS-REVIEW', f'{base_reason} Human evaluation missing - cannot determine reasoning quality.'
    if loc and exp:
        # Correct diagnosis (Loc=T, Exp=T) but correction plan failed → RSN-PLAN
        return 'RSN-PLAN', f'{base_reason} Diagnosis correct but correction plan was flawed/incomplete.'
    # Any other combination: diagnosis itself was wrong → RSN-DIAG
    detail = ''
    if not loc:
        detail += ' Wrong step identified (Loc=F).'
    if not exp:
        detail += ' Wrong or insufficient explanation (Exp=F).'
    return 'RSN-DIAG', (base_reason + detail).strip()


def suggest_taxonomy(row):
    """Return (code, reason) suggestion for one failed episode.

    Root-cause hierarchy (checked in order):
      PER-MISSING  → state never sensed at all
      PER-SPATIAL  → sensed but wrong spatial relationship assigned
      CTX-HIDDEN   → in L1 but absent from L2
      CTX-AFFORD   → in L2 but too generic to ground
      RSN-DIAG     → LLM had info, diagnosed wrong
      RSN-PLAN     → diagnosis correct, correction plan wrong
      EXEC-SIM     → physics/simulator failure
    """
    gt = str(row.get('r_gt_failure_reason', '')).lower()
    l1 = str(row.get('l1_summary', '')).lower()
    l2 = str(row.get('l2_summary', '')).lower()
    gt_type = str(row.get('r_gt_error_type', ''))
    loc = bool(row.get('Loc', False))
    exp = bool(row.get('Exp', False))
    human_score = row.get('human_score', 0)
    exp_unknown = pd.isna(human_score)
    succ_cond = str(row.get('success_condition', ''))
    co_plan = bool(row.get('c_success', False))

    # ── 1. EXECUTION FAILURE (simulator/physics) ────────────────────────────
    # Only EXEC-SIM when LLM was right (Loc=T, Exp=T) but sim/correction still failed
    if gt_type == 'execution' and loc and exp and not co_plan:
        return 'EXEC-SIM', ('GT=execution, LLM correctly diagnosed (Loc=T, Exp=T), '
                            'but correction also failed - likely a simulator/physics constraint.')

    # ── 2. BLOCKING / OCCLUSION ─────────────────────────────────────────────
    if any(w in gt for w in ['blocking', 'occluding', 'blocked']):
        obj = _extract_blocking_object(gt)

        if any(w in l2 for w in ['blocking', 'occluding']):
            return _rsn_from_loc_exp(loc, exp, exp_unknown,
                'Blocking/occlusion explicitly visible in L2.')

        if obj:
            positional_l2 = any(p in l2 for p in [
                f'{obj} is on top of', f'on top of {obj}',
                f'is below {obj}', f'is under {obj}',
            ])
            if positional_l2:
                return 'PER-SPATIAL', (
                    f'Object "{obj}" present in L2 with a positional relationship, but '
                    'perception assigned the wrong spatial description - blocking not conveyed.')

            if obj in l2:
                return 'PER-SPATIAL', (
                    f'Object "{obj}" present in L2 but spatial description does not convey '
                    'blocking/obstruction - perception mis-labelled the relationship.')

        if any(w in l1 for w in ['blocking', 'occluding']) or (obj and obj in l1):
            return 'CTX-HIDDEN', (
                'Blocking info captured in L1 (non-action frames) but absent from L2 - '
                'LLM was never shown this context.')

        return 'PER-MISSING', 'Blocking object/relationship not captured in any observation.'

    # ── 3. DIRTY STATE ──────────────────────────────────────────────────────
    if 'dirty' in gt:
        if 'dirty' in l2:
            return _rsn_from_loc_exp(loc, exp, exp_unknown, 'Dirty state visible in L2.')
        elif 'dirty' in l1:
            return 'CTX-HIDDEN', 'Dirty state captured in L1 but absent from L2 - LLM never saw it.'
        return 'PER-MISSING', 'Dirty state not captured in any observation.'

    # ── 4. DROP / CRACK EVENTS ──────────────────────────────────────────────
    if _re.search(r'\bdrop', gt):
        drop_cues = ['drop', 'crack', 'shatter']
        if any(w in l2 for w in drop_cues):
            return _rsn_from_loc_exp(loc, exp, exp_unknown, 'Drop/crack cue visible in L2.')
        elif any(w in l1 for w in drop_cues):
            return 'CTX-HIDDEN', 'Drop/crack cue in L1 but absent from L2 - LLM never saw it.'
        return 'PER-MISSING', 'Drop event not captured - no cue in any observation.'

    # ── 5. PRE-EXISTING STATE ────────────────────────────────────────────────
    if any(w in gt for w in ['already filled', 'already inside', 'already occupied']):
        state_kws = ['filled', 'inside']
        if any(w in l2 for w in state_kws):
            return _rsn_from_loc_exp(loc, exp, exp_unknown, 'Pre-existing state visible in L2.')
        elif any(w in l1 for w in state_kws):
            return 'CTX-HIDDEN', 'Pre-existing state in L1 but absent from L2.'
        return 'PER-MISSING', 'Pre-existing state not captured in observations.'

    # ── 6. PERCEPTION MISIDENTIFICATION ─────────────────────────────────────
    if 'mis-identified' in gt or 'wrong perception' in gt:
        return 'PER-SPATIAL', ('GT explicitly indicates perception assigned the wrong '
                               'object identity or spatial relationship.')

    # ── 7. AFFORDANCE / GROUNDING GAP ───────────────────────────────────────
    if (_re.search(r'should (?:not )?use .+? instead of', gt)
            or ('cannot be put' in gt and _has_generic_success_condition(succ_cond))
            or 'ambiguous' in gt):
        if _has_generic_success_condition(succ_cond) or 'ambiguous' in gt:
            return 'CTX-AFFORD', (
                f'Success condition or task spec is too generic/ambiguous ("{succ_cond[:80]}") '
                '- LLM cannot ground it to the specific required object or precondition.')

    # ── 8. WRONG SEQUENCING / ORDERING ──────────────────────────────────────
    if 'wrong order' in gt or ('before' in gt and any(w in gt for w in
            ['toggle', 'turn', 'close', 'open', 'slice', 'pick'])):
        if loc and exp:
            return 'RSN-PLAN', 'Correct diagnosis, but correction plan has wrong step ordering.'
        return _rsn_from_loc_exp(loc, exp, exp_unknown, 'GT involves action ordering error.')

    # ── 9. MISSING / SKIPPED STEPS ──────────────────────────────────────────
    if 'missing' in gt or ('never' in gt and any(w in gt for w in
            ['picked', 'opened', 'cracked', 'sliced', 'cleaned', 'executed',
             'removed', 'emptied', 'poured'])):
        return _rsn_from_loc_exp(loc, exp, exp_unknown, 'GT involves missing/skipped action step(s).')

    # ── 10. WRONG ACTION / OBJECT / PLAN ────────────────────────────────────
    if any(w in gt for w in ['wrong plan', 'wrong execution', 'should use',
                              'should have', 'instead of', 'wrong']):
        return _rsn_from_loc_exp(loc, exp, exp_unknown, 'GT indicates wrong action or object usage.')

    # ── 11. FAILED ACTION ───────────────────────────────────────────────────
    if gt.startswith('failed') or 'failed to' in gt:
        return _rsn_from_loc_exp(loc, exp, exp_unknown, 'Action execution reported as failed.')

    # ── 12. FALLBACK ────────────────────────────────────────────────────────
    if exp_unknown:
        return 'NEEDS-REVIEW', 'Human evaluation score missing - cannot determine reasoning quality.'

    return 'RSN-DIAG', 'LLM failed to correctly identify and explain the failure despite available context.'


# Apply suggestions to all failed episodes
suggestions = failed.apply(suggest_taxonomy, axis=1, result_type='expand')
failed['suggested_code'] = suggestions[0]
failed['suggested_reason'] = suggestions[1]

# Summary
print("Auto-suggestion distribution:")
print(failed['suggested_code'].value_counts().to_string())
print(f"\nTotal: {len(failed)} episodes suggested")


## Round 1 - Quick Triage

In [ ]:
# Helpers for display
def _fmt_plan(plan):
    """Format a plan list as an HTML ordered list."""
    if not plan:
        return '<em style="color:#999;">-</em>'
    items = ''.join(f'<li style="font-family:monospace; font-size:12px;">{str(s)}</li>' for s in plan)
    return f'<ol style="margin:4px 0 0 16px; padding:0;">{items}</ol>'


def _fmt_plan_diff(plan, ref_set):
    """Format plan highlighting steps NOT in ref_set (i.e. added by LLM re-plan)."""
    if not plan:
        return '<em style="color:#999;">-</em>'
    items = []
    for s in plan:
        key = str(s).lower()
        if key not in ref_set:
            items.append(
                f'<li style="font-family:monospace; font-size:12px; '
                f'background:#fff3cd; border-left:3px solid #ffc107; padding-left:4px;">'
                f'&#x2795; {s}</li>'
            )
        else:
            items.append(f'<li style="font-family:monospace; font-size:12px;">{s}</li>')
    return f'<ol style="margin:4px 0 0 16px; padding:0;">{"".join(items)}</ol>'


def display_episode_summary(row):
    """Display a compact summary card for one episode with suggestion and full L1/L2 summaries."""
    l1_summary = row.get('l1_summary', '') or ''
    l2_summary = row.get('l2_summary', '') or ''
    suggested_code = row.get('suggested_code', '')
    suggested_reason = row.get('suggested_reason', '')

    gt_step = row['r_gt_failure_step']
    pred_step = row['r_pred_failure_step']

    # rp_task_plan: original task action sequence from task.json (stored in replan_dict)
    # rp_llm_plan:  LLM-generated correction plan (parsed, stored in replan_dict)
    # Fallback to gt_plan/rp_plan for older result files
    gt_plan  = row.get('rp_task_plan') or row.get('gt_plan') or []
    llm_plan = row.get('rp_llm_plan') or row.get('rp_plan') or []

    # Normalize gt_plan for diff highlighting (strip navigate_to_obj steps, lowercase)
    gt_norm = set(str(s).lower() for s in gt_plan)
    llm_plan_html = _fmt_plan_diff(llm_plan, gt_norm)

    # Description tooltip for suggested code
    code_desc = TAXONOMY_DESCRIPTIONS.get(str(suggested_code), '')

    html = f"""
    <div style="border:1px solid #ccc; padding:10px; margin:5px 0; border-radius:6px; background:#f9f9f9;">
      <h3 style="margin:0 0 6px 0;">&#x1F50D; {row['episode']}</h3>
      <table style="width:100%; font-size:13px;">
        <tr><td style="width:160px;"><b>Task</b></td><td>{row['task']}</td></tr>
        <tr><td><b>GT error type</b></td><td>{row['r_gt_error_type']}</td></tr>
        <tr><td><b>Pred error type</b></td><td>{row['r_error_type']}</td></tr>
        <tr><td><b>Loc / Exp</b></td><td>{'&#x2705;' if row['Loc'] else '&#x274C;'} / {'&#x2705;' if row['Exp'] else '&#x274C;'}</td></tr>
        <tr><td><b>GT failure step</b></td><td>{gt_step}</td></tr>
        <tr><td><b>Pred failure step</b></td><td>{pred_step}</td></tr>
        <tr><td colspan="2"><hr style="margin:4px 0;"></td></tr>
        <tr><td><b>GT reason</b></td><td style="white-space:pre-wrap;">{row['r_gt_failure_reason']}</td></tr>
        <tr><td><b>Pred reason</b></td><td style="white-space:pre-wrap;">{row['r_pred_failure_reason']}</td></tr>
      </table>
      <div style="margin-top:8px; padding:8px; background:#e8f4fd; border-left:4px solid #2196F3; border-radius:4px;">
        <b>&#x1F4A1; Suggested: <code>{suggested_code}</code></b>
        <span style="font-size:11px; color:#666; margin-left:8px;">{code_desc}</span><br>
        <span style="font-size:12px; color:#555;">{suggested_reason}</span>
      </div>
      <!-- ── Plans ── -->
      <details open><summary style="cursor:pointer; margin-top:8px; font-weight:bold;">
        &#x1F4CB; Plans (task original &#x2192; LLM re-plan)</summary>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-top:6px;">
          <div>
            <div style="font-size:12px; font-weight:bold; color:#555; margin-bottom:2px;">
              Task original plan - task.json ({len(gt_plan)} steps)</div>
            {_fmt_plan(gt_plan)}
          </div>
          <div>
            <div style="font-size:12px; font-weight:bold; color:#1565C0; margin-bottom:2px;">
              LLM re-plan - replan_dict ({len(llm_plan)} steps)
              <span style="font-weight:normal; color:#888;">&#x2795; = step not in GT</span></div>
            {llm_plan_html}
          </div>
        </div>
      </details>
      <details><summary style="cursor:pointer; margin-top:8px;"><b>L2 Summary (subgoal-level)</b></summary>
        <pre style="font-size:11px; max-height:300px; overflow-y:auto; background:#fff; padding:6px;">{l2_summary}</pre>
      </details>
      <details><summary style="cursor:pointer; margin-top:4px;"><b>L1 Summary (step-level, full)</b></summary>
        <pre style="font-size:11px; max-height:400px; overflow-y:auto; background:#fff; padding:6px;">{l1_summary}</pre>
      </details>
    </div>
    """
    display(HTML(html))


In [ ]:
# Load custom codes persisted from previous sessions

_custom = {}
if os.path.exists(CUSTOM_CODES_PATH):
    try:
        with open(CUSTOM_CODES_PATH) as _f:
            _custom = json.load(_f)
    except Exception:
        _custom = {}
for _code, _desc in _custom.items():
    if _code not in TAXONOMY_CODES:
        TAXONOMY_CODES.append(_code)
        TAXONOMY_DESCRIPTIONS[_code] = _desc
print(f"Custom codes loaded: {list(_custom.keys()) or 'none'}")

# If a previous triage file exists, resume from it
if os.path.exists(OUTPUT_CSV):
    prev = pd.read_csv(OUTPUT_CSV)
    # Merge previous labels into failed
    failed = failed.merge(prev[['episode','taxonomy_code','notes']], on='episode', how='left')
    print(f"Loaded {prev['taxonomy_code'].notna().sum()} previous labels from {OUTPUT_CSV}")
else:
    failed['taxonomy_code'] = ''
    failed['notes'] = ''
    print("Starting fresh - no previous labels found.")


In [ ]:
# Interactive triage widget
# Shows one episode at a time. Dropdown is pre-populated with the auto-suggestion.
# Labels are auto-saved to CSV on every navigation.

triage_idx = [0]  # mutable counter

# Columns to save; gt_plan kept as fallback for older result files
SAVE_COLS = ['episode', 'task', 'r_gt_error_type', 'r_error_type', 'Loc', 'Exp',
             'r_gt_failure_reason', 'r_pred_failure_reason',
             'r_gt_failure_step', 'r_pred_failure_step',
             'rp_task_plan', 'rp_llm_plan', 'gt_plan',
             'taxonomy_code', 'notes']

out = widgets.Output()

# Main label controls
dropdown = widgets.Dropdown(
    options=TAXONOMY_CODES,
    description='Category:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='260px'),
)
notes_box = widgets.Text(
    placeholder='Optional note',
    description='Note:',
    layout=widgets.Layout(width='55%'),
)
# Description label that updates when dropdown changes
desc_label = widgets.HTML(value='')

def _on_dropdown_change(change):
    code = change['new']
    desc = TAXONOMY_DESCRIPTIONS.get(code, '')
    desc_label.value = f'<span style="font-size:11px; color:#555; margin-left:4px;">{desc}</span>'

dropdown.observe(_on_dropdown_change, names='value')
_on_dropdown_change({'new': dropdown.value})  # initialise

progress_label = widgets.Label()

# Navigation / action buttons
btn_prev          = widgets.Button(description='◀ Prev')
btn_next          = widgets.Button(description='Next ▶')
btn_skip_labelled = widgets.Button(description='Skip to unlabelled ⏭', button_style='info')
btn_details       = widgets.Button(description='🔬 Load details', button_style='warning')
btn_accept        = widgets.Button(description='✅ Accept suggestion', button_style='primary')
btn_prompts       = widgets.Button(description='📋 View Prompts', button_style='info')

# Add-label-on-the-go controls
new_code_input = widgets.Text(
    placeholder='NEW-CODE',
    description='New label:',
    layout=widgets.Layout(width='200px'),
    style={'description_width': '80px'},
)
new_desc_input = widgets.Text(
    placeholder='One-line description',
    description='Description:',
    layout=widgets.Layout(width='50%'),
    style={'description_width': '90px'},
)
btn_add_code = widgets.Button(description='➕ Add label', button_style='success',
                               layout=widgets.Layout(width='120px'))
add_code_out = widgets.Output()

def _refresh():
    idx = triage_idx[0]
    row = failed.iloc[idx]
    with out:
        clear_output(wait=True)
        display_episode_summary(row)
    cur_code = failed.at[idx, 'taxonomy_code']
    if cur_code and str(cur_code).strip() and cur_code in TAXONOMY_CODES:
        dropdown.value = cur_code
    else:
        suggested = failed.at[idx, 'suggested_code']
        if suggested and suggested in TAXONOMY_CODES:
            dropdown.value = suggested
        else:
            dropdown.value = TAXONOMY_CODES[0]
    notes_box.value = (str(failed.at[idx, 'notes'])
                       if pd.notna(failed.at[idx, 'notes']) and str(failed.at[idx, 'notes']).strip()
                       else '')
    labelled = (failed['taxonomy_code'].astype(str).str.len() > 0).sum()
    progress_label.value = (f"Episode {idx + 1} / {len(failed)}  |  "
                             f"Labelled: {labelled} / {len(failed)}  |  ✅ auto-saved")

def _store_and_save():
    idx = triage_idx[0]
    failed.at[idx, 'taxonomy_code'] = dropdown.value
    failed.at[idx, 'notes'] = notes_box.value
    cols_to_save = [c for c in SAVE_COLS if c in failed.columns]
    failed[cols_to_save].to_csv(OUTPUT_CSV, index=False)

def _on_prev(_):
    _store_and_save()
    triage_idx[0] = max(0, triage_idx[0] - 1)
    _refresh()

def _on_next(_):
    _store_and_save()
    triage_idx[0] = min(len(failed) - 1, triage_idx[0] + 1)
    _refresh()

def _on_accept(_):
    idx = triage_idx[0]
    suggested_code = failed.at[idx, 'suggested_code']
    suggested_reason = failed.at[idx, 'suggested_reason']
    if suggested_code and suggested_code in TAXONOMY_CODES:
        dropdown.value = suggested_code
    notes_box.value = str(suggested_reason) if suggested_reason else ''
    _store_and_save()
    triage_idx[0] = min(len(failed) - 1, triage_idx[0] + 1)
    _refresh()

def _on_skip(_):
    _store_and_save()
    for i in range(triage_idx[0] + 1, len(failed)):
        code = failed.at[i, 'taxonomy_code']
        if not code or (isinstance(code, float) and np.isnan(code)) or str(code).strip() == '':
            triage_idx[0] = i
            _refresh()
            return
    progress_label.value = "All episodes are labelled!"

def _on_details(_):
    """Show the current episode's raw record fields for quick inspection."""
    idx = triage_idx[0]
    row = failed.iloc[idx]
    detail_keys = [
        'episode', 'task', 'success_condition',
        'r_gt_error_type', 'r_error_type',
        'r_gt_failure_step', 'r_pred_failure_step',
        'r_gt_failure_reason', 'r_pred_failure_reason',
        'suggested_code', 'suggested_reason',
        'Loc', 'Exp', 'human_score',
        'rp_task_plan', 'rp_llm_plan', 'gt_plan',
    ]
    with out:
        clear_output(wait=True)
        display_episode_summary(row)
        details = []
        for key in detail_keys:
            if key not in row.index:
                continue
            value = row[key]
            if isinstance(value, float) and pd.isna(value):
                value = ''
            if isinstance(value, (list, dict)):
                formatted = json.dumps(value, indent=2)
            else:
                formatted = str(value)
            details.append(
                f'<tr><td style="vertical-align:top; width:180px;"><b>{key}</b></td>'
                f'<td><pre style="margin:0; white-space:pre-wrap; font-size:11px;">{formatted}</pre></td></tr>'
            )
        display(HTML(
            '<div style="border:1px solid #d0d0d0; border-radius:6px; padding:8px; margin-top:8px; background:#fff;">'
            '<b>Episode details</b>'
            '<table style="width:100%; margin-top:6px; font-size:12px;">'
            + ''.join(details) + '</table></div>'
        ))

def _on_add_code(_):
    """Add a new custom taxonomy code on the fly and persist it."""
    with add_code_out:
        clear_output(wait=True)
        code = new_code_input.value.strip().upper()
        desc = new_desc_input.value.strip()
        if not code:
            print("⚠️  Code name is empty.")
            return
        if code in TAXONOMY_CODES:
            print(f"⚠️  '{code}' already exists.")
            return
        TAXONOMY_CODES.append(code)
        TAXONOMY_DESCRIPTIONS[code] = desc
        dropdown.options = TAXONOMY_CODES
        # Persist to JSON
        existing = {}
        if os.path.exists(CUSTOM_CODES_PATH):
            try:
                with open(CUSTOM_CODES_PATH) as _f:
                    existing = json.load(_f)
            except Exception:
                pass
        existing[code] = desc
        with open(CUSTOM_CODES_PATH, 'w') as _f:
            json.dump(existing, _f, indent=2)
        new_code_input.value = ''
        new_desc_input.value = ''
        print(f"✅ Added '{code}': {desc}")

btn_prev.on_click(_on_prev)
btn_next.on_click(_on_next)
btn_skip_labelled.on_click(_on_skip)
btn_details.on_click(_on_details)
btn_accept.on_click(_on_accept)
btn_add_code.on_click(_on_add_code)

def _on_view_prompts(_):
    """Show all LLM prompt/response pairs for the current episode in a collapsible output."""
    idx = triage_idx[0]
    row = failed.iloc[idx]
    prompts_log = row.get('prompts_log') if hasattr(row, 'get') else None
    # If stored as a JSON string (happens after CSV round-trip), decode it
    if isinstance(prompts_log, str):
        try:
            prompts_log = json.loads(prompts_log)
        except Exception:
            prompts_log = None
    with out:
        if not prompts_log:
            print("⚠️  No prompt log available for this episode (run with new pipeline to capture).")
            return
        parts = []
        for entry in prompts_log:
            call_name = entry.get('call', '?')
            prompt = entry.get('prompt', {})
            response = entry.get('response', '')
            sys_txt = str(prompt.get('system', '')).replace('<', '&lt;').replace('>', '&gt;')[:300]
            usr_txt = str(prompt.get('user', '')).replace('<', '&lt;').replace('>', '&gt;')
            rsp_txt = str(response).replace('<', '&lt;').replace('>', '&gt;')
            parts.append(
                f'<details style="margin:4px 0; border:1px solid #ccc; border-radius:4px; padding:4px;">'
                f'<summary style="cursor:pointer; font-weight:bold; color:#1565C0;">'
                f'&#x1F4AC; {call_name}</summary>'
                f'<div style="margin:6px;">'
                f'<div style="font-size:11px; color:#888; margin-bottom:2px;">System (truncated):</div>'
                f'<pre style="font-size:10px; background:#f5f5f5; padding:4px; max-height:80px; overflow-y:auto;">{sys_txt}...</pre>'
                f'<div style="font-size:11px; color:#888; margin:4px 0 2px;">User prompt:</div>'
                f'<pre style="font-size:10px; background:#e8f4fd; padding:4px; max-height:200px; overflow-y:auto;">{usr_txt}</pre>'
                f'<div style="font-size:11px; color:#888; margin:4px 0 2px;">Response:</div>'
                f'<pre style="font-size:10px; background:#e8f5e9; padding:4px; max-height:150px; overflow-y:auto;">{rsp_txt}</pre>'
                f'</div></details>'
            )
        display(HTML(
            f'<div style="border:1px solid #2196F3; border-radius:6px; padding:8px; margin:4px 0;">'
            f'<b>&#x1F4CB; Prompt log - {row["episode"]} ({len(prompts_log)} LLM calls)</b>'
            + ''.join(parts) + '</div>'
        ))

btn_prompts.on_click(_on_view_prompts)

nav_box   = widgets.HBox([btn_prev, btn_next, btn_accept, btn_skip_labelled,
                           btn_details, btn_prompts])
input_box = widgets.HBox([dropdown, notes_box])
desc_box  = widgets.HBox([desc_label])
add_box   = widgets.HBox([new_code_input, new_desc_input, btn_add_code])
display(widgets.VBox([progress_label, out, input_box, desc_box, nav_box,
                      widgets.HTML('<hr style="margin:6px 0;">'),
                      widgets.HTML('<b style="font-size:12px;">Add custom label on the go:</b>'),
                      add_box, add_code_out]))
_refresh()

---
## Save Round 1 Results

Run this after you finish triage to make sure everything is persisted.

In [ ]:

# Force-save current state and print distribution summary
_store_and_save()
print(f"Saved {(failed['taxonomy_code'].astype(str).str.len() > 0).sum()}/{len(failed)} labelled episodes to {OUTPUT_CSV}")
print("\nTaxonomy distribution so far:")
print(failed['taxonomy_code'].value_counts().to_string())


## Summary Statistics

In [ ]:
# Reload final CSV
final = pd.read_csv(OUTPUT_CSV)
print(f"Total labelled: {final['taxonomy_code'].notna().sum()} / {len(final)}")
print(f"Still NEEDS-REVIEW: {(final['taxonomy_code'] == 'NEEDS-REVIEW').sum()}")
print()

# Category-level labels for grouping (updated taxonomy)
CATEGORY_MAP = {
    'PER-MISSING': 'Perception',
    'PER-SPATIAL': 'Perception',
    'PER-AUDIO': 'Perception',
    'CTX-HIDDEN': 'Context / Affordance',
    'CTX-AFFORD': 'Context / Affordance',
    'RSN-DIAG': 'LLM Reasoning',
    'RSN-PLAN': 'LLM Reasoning',
    'TR-EMBED': 'Plan Translation',
    'EXEC-SIM': 'Eval / Simulator',
    'OTHER': 'Other',
    'NEEDS-REVIEW': 'Review',
}
final['category'] = final['taxonomy_code'].map(CATEGORY_MAP)

# Sub-type distribution
code_counts = final['taxonomy_code'].value_counts()
print("Sub-type distribution:")
print(code_counts.to_string())
print()

# Category distribution
cat_counts = final['category'].value_counts()
print("Category distribution:")
print(cat_counts.to_string())

In [ ]:
# ── Visualization ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart: sub-type codes
code_counts.plot.bar(ax=axes[0], color='#4c72b0', edgecolor='white')
axes[0].set_title('Error Sub-type Distribution', fontsize=15)
axes[0].set_ylabel('Count', fontsize=15)
for p in axes[0].patches:
    height = p.get_height()
    if height > 0:
        axes[0].text(p.get_x() + p.get_width() / 2, height + 0.1, f'{int(height)}',
                     ha='center', va='bottom', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)
    
# Pie chart: high-level categories
cat_colors = {
    'Perception': '#4c72b0', 'LLM Reasoning': '#e07b39',
    'Context / Temporal': '#55a868', 'Plan Translation': '#c44e52',
    'Eval / Simulator': '#8172b2', 'Other': '#ccb974', 'Review': '#999999',
}
colors = [cat_colors.get(c, '#aaa') for c in cat_counts.index]
cat_counts.plot.pie(ax=axes[1], colors=colors, autopct='%1.0f%%', startangle=140, textprops={'fontsize': 12})
axes[1].tick_params(axis='x', labelsize=15)
axes[1].set_title('Error Category Distribution', fontsize=15)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Taxonomy distribution by task
crosstab = pd.crosstab(final['task'], final['taxonomy_code'], margins=True)
crosstab = crosstab.drop(index='All', columns='All')
crosstab.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
plt.title('Error Taxonomy Distribution by Task', fontsize=14, fontweight='bold')
plt.xlabel('Task')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Taxonomy Code', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Taxonomy distribution by Loc/Exp groups
final['group'] = final.apply(
    lambda r: f"Loc={'T' if r['Loc'] else 'F'} Exp={'T' if r['Exp'] else 'F'}", axis=1
)
crosstab2 = pd.crosstab(final['group'], final['taxonomy_code'], margins=True)

crosstab2 = crosstab2.drop(index='All', columns='All')
crosstab2.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
plt.title('Error Taxonomy Distribution by Loc/Exp Group', fontsize=14, fontweight='bold')
plt.xlabel('Loc/Exp Group')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Taxonomy Code', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── Export final taxonomy with all episodes (including successful ones) for downstream use ──
full_export = run_df[['episode', 'task', 'c_success']].copy()
full_export = full_export.merge(final[['episode', 'taxonomy_code', 'category', 'notes']], on='episode', how='left')
full_export.loc[full_export['c_success'] == True, 'taxonomy_code'] = 'SUCCESS'
full_export.loc[full_export['c_success'] == True, 'category'] = 'Success'
full_export.to_csv(os.path.join(RESULTS_PATH, "error_taxonomy_full.csv"), index=False)
print(f"Full taxonomy exported to {RESULTS_PATH}error_taxonomy_full.csv")
print(full_export['taxonomy_code'].value_counts().to_string())